# 蛋白质可视化原型软件
本 Notebook 在 **Python 3.11** 环境中使用 **NGLView** 与 **ipywidgets** 构建一个可交互的蛋白质可视化原型。

> 功能概览
- 加载单个 PDB 结构
- 显示主链、侧链、球棒、分子表面
- 解析 MMPBSA 输出中的 `DELTA TOTAL` 残基能量
- 将能量映射到残基颜色：低能量→红色，高能量→蓝色
- 通过控件调整阈值、切换显示模式、交互选残基

> 说明
- 所有示例数据均内嵌在 Notebook 中，不生成额外文件。
- 若本地环境缺少依赖，请先执行下方安装单元。
- 建议按顺序运行全部单元。

## 1. 安装与导入依赖
本节安装并导入所需依赖，包括：
- `nglview`：3D 分子可视化
- `ipywidgets`：交互控件
- `numpy` / `pandas`：数据处理
- `biopython`：可选结构解析支持

> 如果你已经安装好这些库，可以直接跳到下一单元。

In [ ]:
# 如需安装依赖，请取消下一行注释后执行
# %pip install -U nglview ipywidgets numpy pandas biopython matplotlib

import io
import re
import math
import tempfile
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    import nglview as nv
except ImportError as exc:
    raise ImportError(
        "未安装 nglview。请先运行 %pip install -U nglview ipywidgets numpy pandas biopython matplotlib"
    ) from exc

try:
    import ipywidgets
except ImportError as exc:
    raise ImportError("未安装 ipywidgets。请先安装后重试。") from exc

# 某些环境下需要启用小部件扩展；在 JupyterLab 4 / VS Code Notebook 中通常可直接使用。
print(f'nglview version: {nv.__version__}')
print(f'ipywidgets version: {widgets.__version__}')

## 2. 定义示例 PDB 数据与 MMPBSA 解析输入
本节以内嵌字符串方式提供：
- 一个可直接演示的 PDB 片段
- 一个模拟的 MMPBSA `DELTA TOTAL` 输出

> 这样可以确保 Notebook 独立运行，不依赖外部文件。后续也支持上传你自己的 PDB 与 MMPBSA 文本。

In [ ]:
PDB_TEXT = '''
ATOM      1  N   ALA A   1      11.104  13.207   7.115  1.00 20.00           N  
ATOM      2  CA  ALA A   1      12.560  13.404   7.275  1.00 20.00           C  
ATOM      3  C   ALA A   1      13.081  12.990   8.657  1.00 20.00           C  
ATOM      4  O   ALA A   1      12.424  12.261   9.405  1.00 20.00           O  
ATOM      5  CB  ALA A   1      13.050  14.853   6.997  1.00 20.00           C  
ATOM      6  N   TYR A   2      14.284  13.461   9.006  1.00 18.00           N  
ATOM      7  CA  TYR A   2      14.893  13.140  10.293  1.00 18.00           C  
ATOM      8  C   TYR A   2      15.287  11.667  10.426  1.00 18.00           C  
ATOM      9  O   TYR A   2      15.329  10.929   9.437  1.00 18.00           O  
ATOM     10  CB  TYR A   2      16.129  14.031  10.515  1.00 18.00           C  
ATOM     11  CG  TYR A   2      15.889  15.501  10.330  1.00 18.00           C  
ATOM     12  CD1 TYR A   2      15.005  16.169  11.183  1.00 18.00           C  
ATOM     13  CD2 TYR A   2      16.548  16.220   9.326  1.00 18.00           C  
ATOM     14  CE1 TYR A   2      14.792  17.521  11.010  1.00 18.00           C  
ATOM     15  CE2 TYR A   2      16.341  17.569   9.146  1.00 18.00           C  
ATOM     16  CZ  TYR A   2      15.463  18.207  10.006  1.00 18.00           C  
ATOM     17  OH  TYR A   2      15.257  19.548   9.827  1.00 18.00           O  
ATOM     18  N   GLU A   3      15.585  11.251  11.662  1.00 16.00           N  
ATOM     19  CA  GLU A   3      15.923   9.859  11.973  1.00 16.00           C  
ATOM     20  C   GLU A   3      17.326   9.470  11.530  1.00 16.00           C  
ATOM     21  O   GLU A   3      18.235  10.274  11.700  1.00 16.00           O  
ATOM     22  CB  GLU A   3      15.702   9.575  13.470  1.00 16.00           C  
ATOM     23  CG  GLU A   3      14.292   9.924  13.934  1.00 16.00           C  
ATOM     24  CD  GLU A   3      14.051   9.611  15.401  1.00 16.00           C  
ATOM     25  OE1 GLU A   3      14.967   9.805  16.221  1.00 16.00           O  
ATOM     26  OE2 GLU A   3      12.941   9.167  15.737  1.00 16.00           O  
ATOM     27  N   LYS A   4      17.493   8.242  10.972  1.00 15.00           N  
ATOM     28  CA  LYS A   4      18.782   7.732  10.487  1.00 15.00           C  
ATOM     29  C   LYS A   4      19.026   6.305  10.970  1.00 15.00           C  
ATOM     30  O   LYS A   4      18.118   5.479  10.987  1.00 15.00           O  
ATOM     31  CB  LYS A   4      18.846   7.806   8.954  1.00 15.00           C  
ATOM     32  CG  LYS A   4      20.226   7.495   8.373  1.00 15.00           C  
ATOM     33  CD  LYS A   4      20.255   7.622   6.858  1.00 15.00           C  
ATOM     34  CE  LYS A   4      21.664   7.359   6.343  1.00 15.00           C  
ATOM     35  NZ  LYS A   4      21.692   7.523   4.869  1.00 15.00           N  
ATOM     36  N   VAL A   5      20.265   6.026  11.356  1.00 17.00           N  
ATOM     37  CA  VAL A   5      20.671   4.705  11.865  1.00 17.00           C  
ATOM     38  C   VAL A   5      21.337   3.880  10.777  1.00 17.00           C  
ATOM     39  O   VAL A   5      22.239   4.344  10.081  1.00 17.00           O  
ATOM     40  CB  VAL A   5      21.626   4.815  13.080  1.00 17.00           C  
ATOM     41  CG1 VAL A   5      22.969   5.463  12.689  1.00 17.00           C  
ATOM     42  CG2 VAL A   5      21.893   3.443  13.683  1.00 17.00           C  
TER
END
'''

MMPBSA_TEXT = '''
Resid  Chain Residue    DELTA TOTAL
1      A     ALA       -12.5
2      A     TYR        -7.2
3      A     GLU         1.8
4      A     LYS         6.4
5      A     VAL        11.1
'''

print(PDB_TEXT.splitlines()[0])
print(MMPBSA_TEXT)

## 3. 实现 PDB 残基索引与原子记录解析
这里实现一个轻量级 PDB 解析器，用于：
- 提取链 ID、残基编号、残基名、原子名
- 区分主链与侧链原子
- 构建残基到 NGL 选择器的映射

> 这样可以不依赖外部文件解析流程，便于在 Notebook 中直接演示。

In [ ]:
BACKBONE_ATOMS = {"N", "CA", "C", "O", "OXT"}

@dataclass(frozen=True)
class ResidueKey:
    chain: str
    resid: int
    resname: str
    
    def ngl_selector(self) -> str:
        return f":{self.chain} and {self.resid}"

def parse_pdb_atoms(pdb_text: str) -> pd.DataFrame:
    """解析 PDB 文本中的 ATOM/HETATM 记录，返回原子级 DataFrame。"""
    records = []
    for line in pdb_text.splitlines():
        if not line.startswith(("ATOM", "HETATM")):
            continue
        atom_name = line[12:16].strip()
        resname = line[17:20].strip()
        chain = (line[21].strip() or "A")
        resid = int(line[22:26].strip())
        x = float(line[30:38].strip())
        y = float(line[38:46].strip())
        z = float(line[46:54].strip())
        element = line[76:78].strip() or atom_name[0]
        atom_type = "backbone" if atom_name in BACKBONE_ATOMS else "sidechain"
        records.append(
            {
                "chain": chain,
                "resid": resid,
                "resname": resname,
                "atom_name": atom_name,
                "element": element,
                "x": x,
                "y": y,
                "z": z,
                "atom_type": atom_type,
            }
        )
    return pd.DataFrame(records)

def build_residue_index(atom_df: pd.DataFrame):
    """建立残基索引与常用选择器映射。"""
    residue_index = {}
    residue_rows = atom_df[["chain", "resid", "resname"]].drop_duplicates()
    for row in residue_rows.itertuples(index=False):
        key = ResidueKey(chain=row.chain, resid=int(row.resid), resname=row.resname)
        residue_index[key] = {
            "all": key.ngl_selector(),
            "backbone": f"({key.ngl_selector()}) and backbone",
            "sidechain": f"({key.ngl_selector()}) and sidechain",
            "ball_stick": key.ngl_selector(),
        }
    return residue_index

atom_df = parse_pdb_atoms(PDB_TEXT)
residue_index = build_residue_index(atom_df)

display(atom_df.head())
print(f"原子数: {len(atom_df)}")
print(f"残基数: {len(residue_index)}")
print("示例残基选择器:")
for key, selectors in list(residue_index.items())[:2]:
    print(key, selectors)

## 4. 实现 MMPBSA `DELTA TOTAL` 输出解析
这里编写一个健壮的解析器，从文本中提取按残基统计的 `DELTA TOTAL` 能量值。

> 支持：
- 忽略空行和标题行
- 兼容常见空白分隔格式
- 自动去重，重复残基保留最后一项
- 对异常行进行容错处理

In [ ]:
def parse_mmpbsa_delta_total(text: str) -> pd.DataFrame:
    """从 MMPBSA 输出文本中提取 DELTA TOTAL 残基能量。
    期望格式示例：
        Resid  Chain Residue    DELTA TOTAL
        1      A     ALA       -12.5
    """
    rows = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if "DELTA TOTAL" in line.upper() and any(ch.isalpha() for ch in line):
            continue
        parts = re.split(r"\s+", line)
        if len(parts) < 4:
            continue
        try:
            resid = int(parts[0])
            chain = parts[1]
            resname = parts[2]
            energy = float(parts[-1])
        except (ValueError, IndexError):
            continue
        rows.append(
            {
                "chain": chain,
                "resid": resid,
                "resname": resname,
                "delta_total": energy,
            }
        )

    energy_df = pd.DataFrame(rows)
    if energy_df.empty:
        raise ValueError("未能从 MMPBSA 文本中解析到 DELTA TOTAL 残基能量。")

    energy_df = energy_df.drop_duplicates(subset=["chain", "resid"], keep="last")
    energy_df = energy_df.sort_values(["chain", "resid"]).reset_index(drop=True)
    return energy_df

energy_df = parse_mmpbsa_delta_total(MMPBSA_TEXT)
display(energy_df)

## 5. 实现能量归一化与残基颜色映射
本节将残基能量映射到颜色：
- **低能量**：红色
- **高能量**：蓝色

> 我们同时加入阈值过滤逻辑：
- 满足阈值的残基使用热力颜色显示
- 不满足阈值的残基使用中性灰色或更高透明度

In [ ]:
def normalize_energy(values: pd.Series) -> pd.Series:
    """将能量线性归一化到 [0, 1]。"""
    vmin = values.min()
    vmax = values.max()
    if math.isclose(vmin, vmax):
        return pd.Series(np.full(len(values), 0.5), index=values.index)
    return (values - vmin) / (vmax - vmin)

def rgb_to_hex(rgb):
    r, g, b = [max(0, min(255, int(round(v)))) for v in rgb]
    return f"#{r:02x}{g:02x}{b:02x}"

def energy_to_hex_color(norm_value: float) -> str:
    """红→蓝渐变：0 为红，1 为蓝。"""
    red = np.array([220, 53, 69])
    blue = np.array([13, 110, 253])
    rgb = red + (blue - red) * float(norm_value)
    return rgb_to_hex(rgb)

def prepare_energy_table(atom_df: pd.DataFrame, energy_df: pd.DataFrame) -> pd.DataFrame:
    residue_df = atom_df[["chain", "resid", "resname"]].drop_duplicates()
    merged = residue_df.merge(
        energy_df,
        on=["chain", "resid", "resname"],
        how="left",
    )
    merged["delta_total"] = merged["delta_total"].fillna(0.0)
    merged["norm_energy"] = normalize_energy(merged["delta_total"])
    merged["heat_color"] = merged["norm_energy"].map(energy_to_hex_color)
    merged["residue_label"] = merged.apply(
        lambda r: f"{r['resname']} {r['chain']}{int(r['resid'])}", axis=1
    )
    return merged

energy_table = prepare_energy_table(atom_df, energy_df)
display(energy_table)

plt.figure(figsize=(7, 3))
plt.bar(energy_table["residue_label"], energy_table["delta_total"], color=energy_table["heat_color"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("DELTA TOTAL")
plt.title("MMPBSA 残基能量热力预览")
plt.tight_layout()
plt.show()

## 6. 构建 NGLView 3D 可视化组件
本节创建一个 NGLView 视图对象，并将内嵌 PDB 结构加载到 3D 查看器中。

> 功能：
- 支持旋转、缩放、平移
- 适合在 Notebook 中直接交互查看
- 为后续添加多种表示和颜色热力映射做准备

In [ ]:
def make_temp_pdb_file(pdb_text: str) -> str:
    """将内嵌 PDB 文本写入临时文件，供 NGLView 加载。"""
    tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".pdb", delete=False, encoding="utf-8")
    tmp.write(pdb_text)
    tmp.flush()
    tmp.close()
    return tmp.name

PDB_TEMP_PATH = make_temp_pdb_file(PDB_TEXT)
view = nv.show_file(PDB_TEMP_PATH)
view.layout = widgets.Layout(width="100%", height="650px")
view.center()
view.camera = "orthographic"
view.parameters = {"backgroundColor": "white"}
view

## 7. 添加主链、侧链、球棒与分子表面表示
为同一结构叠加多种表示：
- 主链：`cartoon` / `line`
- 侧链：`licorice`
- 球棒：`ball+stick` 风格近似实现
- 表面：`surface`，可用于近似展示 Van der Waals / SASA 视觉效果

> 后续会通过控件动态切换这些表示的可见性与透明度。

In [ ]:
def reset_base_representations(viewer):
    """清空并重建基础表示层。"""
    viewer.clear_representations()
    viewer.add_cartoon(selection="protein", color="lightgray", opacity=0.85, name="main_cartoon")
    viewer.add_licorice(selection="sidechain", color="silver", opacity=0.85, name="sidechain_licorice")
    viewer.add_ball_and_stick(selection="protein", color="element", opacity=0.35, name="ball_stick")
    viewer.add_surface(selection="protein", color="white", opacity=0.15, name="surface")
    viewer.center()

reset_base_representations(view)
view

## 8. 实现残基选择与高亮交互
本节实现残基选择逻辑：
- 按链 ID 和残基编号定位残基
- 在 3D 视图中高亮该残基
- 同时在信息面板显示对应残基能量

> 选中残基后，会叠加更醒目的球棒或标签层。

In [ ]:
def residue_selector(chain: str, resid: int) -> str:
    return f":{chain} and {int(resid)}"

def lookup_residue_energy(energy_table: pd.DataFrame, chain: str, resid: int):
    hit = energy_table[(energy_table["chain"] == chain) & (energy_table["resid"] == int(resid))]
    if hit.empty:
        return None
    return hit.iloc[0].to_dict()

def show_residue_info(info_output, residue_info):
    with info_output:
        clear_output(wait=True)
        if residue_info is None:
            display(HTML("<b>未找到该残基对应的能量信息。</b>"))
            return
        html = f'''
        <div style="padding:8px 12px;border:1px solid #ddd;border-radius:8px;">
            <h4 style="margin:0 0 8px 0;">残基信息</h4>
            <div><b>残基:</b> {residue_info['resname']} {residue_info['chain']}{int(residue_info['resid'])}</div>
            <div><b>DELTA TOTAL:</b> {residue_info['delta_total']:.3f}</div>
            <div><b>颜色:</b> <span style="display:inline-block;width:14px;height:14px;background:{residue_info['heat_color']};border:1px solid #999;vertical-align:middle;"></span> {residue_info['heat_color']}</div>
        </div>
        '''
        display(HTML(html))

## 9. 实现能量阈值与显示模式控件
这里构建交互控件：
- `FloatSlider`：调整能量阈值
- `ToggleButtons`：切换显示模式
- `Checkbox`：控制主链、侧链、表面是否显示
- `Dropdown` / `IntSlider`：选择残基

> 所有控件将与 3D 可视化联动更新。

In [ ]:
energy_min = float(energy_table["delta_total"].min())
energy_max = float(energy_table["delta_total"].max())

chain_options = sorted(energy_table["chain"].unique().tolist())
default_chain = chain_options[0]
default_resid = int(energy_table[energy_table["chain"] == default_chain]["resid"].min())

threshold_slider = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=max(abs(energy_min), abs(energy_max)),
    step=0.1,
    description="|阈值|",
    readout_format=".1f",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="320px"),
)

display_mode = widgets.ToggleButtons(
    options=[
        ("球棒优先", "ball_stick"),
        ("表面优先", "surface"),
        ("透明模式", "transparent"),
    ],
    value="ball_stick",
    description="显示模式",
    style={"description_width": "70px"},
)

show_backbone = widgets.Checkbox(value=True, description="主链")
show_sidechain = widgets.Checkbox(value=True, description="侧链")
show_surface = widgets.Checkbox(value=True, description="表面")

chain_dropdown = widgets.Dropdown(
    options=chain_options,
    value=default_chain,
    description="链",
    layout=widgets.Layout(width="160px"),
)

residue_slider = widgets.IntSlider(
    value=default_resid,
    min=int(energy_table["resid"].min()),
    max=int(energy_table["resid"].max()),
    step=1,
    description="残基",
    continuous_update=False,
    layout=widgets.Layout(width="320px"),
)

opacity_slider = widgets.FloatSlider(
    value=0.35,
    min=0.0,
    max=1.0,
    step=0.05,
    description="透明度",
    continuous_update=False,
    layout=widgets.Layout(width="320px"),
)

use_uploaded_pdb = widgets.FileUpload(
    accept=".pdb",
    multiple=False,
    description="上传 PDB",
)

use_uploaded_mmpbsa = widgets.FileUpload(
    accept=".txt,.dat,.out,.csv",
    multiple=False,
    description="上传 MMPBSA",
)

reload_button = widgets.Button(description="重新载入数据", button_style="info")
status_output = widgets.Output()
info_output = widgets.Output()

## 10. 联动控件刷新可视化与颜色热力标注
本节实现统一刷新函数，在控件变化时完成：
- 表示层重建
- 阈值筛选
- 能量热力着色
- 选中残基高亮
- 信息面板更新

> 这样可以保证所有界面控件与结构视图保持同步。

In [ ]:
state = {
    "pdb_text": PDB_TEXT,
    "mmpbsa_text": MMPBSA_TEXT,
    "atom_df": atom_df.copy(),
    "energy_df": energy_df.copy(),
    "energy_table": energy_table.copy(),
}

def get_uploaded_text(upload_widget) -> str | None:
    if not upload_widget.value:
        return None
    first_item = next(iter(upload_widget.value.values()))
    content = first_item["content"]
    if isinstance(content, memoryview):
        content = content.tobytes()
    if isinstance(content, bytes):
        return content.decode("utf-8", errors="ignore")
    return str(content)

def build_colored_selection_records(energy_table: pd.DataFrame, threshold: float) -> list[dict]:
    records = []
    for row in energy_table.itertuples(index=False):
        selector = residue_selector(row.chain, int(row.resid))
        active = abs(float(row.delta_total)) >= float(threshold)
        color = row.heat_color if active else "#c7c7c7"
        records.append(
            {
                "selector": selector,
                "color": color,
                "active": active,
                "residue_label": f"{row.resname} {row.chain}{int(row.resid)}",
                "energy": float(row.delta_total),
            }
        )
    return records

def refresh_view(*_):
    current_energy_table = state["energy_table"]
    threshold = float(threshold_slider.value)
    mode = display_mode.value
    selected_chain = chain_dropdown.value
    selected_resid = int(residue_slider.value)
    base_opacity = float(opacity_slider.value)

    view.clear_representations()

    colored_records = build_colored_selection_records(current_energy_table, threshold)

    # 1) 主链表示
    if show_backbone.value:
        for rec in colored_records:
            view.add_cartoon(
                selection=rec["selector"],
                color=rec["color"],
                opacity=0.95 if rec["active"] else max(0.10, base_opacity * 0.35),
            )

    # 2) 侧链表示
    if show_sidechain.value:
        for rec in colored_records:
            view.add_licorice(
                selection=f"({rec['selector']}) and sidechain",
                color=rec["color"],
                opacity=0.90 if rec["active"] else max(0.08, base_opacity * 0.30),
            )

    # 3) 按模式添加覆盖层
    if mode == "ball_stick":
        for rec in colored_records:
            view.add_ball_and_stick(
                selection=rec["selector"],
                color=rec["color"],
                opacity=0.85 if rec["active"] else max(0.05, base_opacity * 0.20),
            )
    elif mode == "surface":
        if show_surface.value:
            for rec in colored_records:
                view.add_surface(
                    selection=rec["selector"],
                    color=rec["color"],
                    opacity=0.45 if rec["active"] else 0.08,
                )
    elif mode == "transparent":
        if show_surface.value:
            view.add_surface(selection="protein", color="white", opacity=min(0.12, base_opacity * 0.4))

    # 4) 补充整体表面层
    if show_surface.value and mode != "surface":
        view.add_surface(selection="protein", color="lightgray", opacity=min(0.18, base_opacity * 0.45))

    # 5) 高亮用户选择的残基
    selected_selector = residue_selector(selected_chain, selected_resid)
    view.add_ball_and_stick(
        selection=selected_selector,
        color="yellow",
        opacity=1.0,
        aspectRatio=1.6,
    )
    view.add_label(
        selection=selected_selector,
        color="black",
        labelType="residue",
        zOffset=2.0,
        attachment="middle-center",
    )

    view.center(selection=selected_selector)
    residue_info = lookup_residue_energy(current_energy_table, selected_chain, selected_resid)
    show_residue_info(info_output, residue_info)

    with status_output:
        clear_output(wait=True)
        active_count = sum(int(rec["active"]) for rec in colored_records)
        total_count = len(colored_records)
        display(HTML(
            f"<div style='padding:6px 10px;background:#f7f7f7;border-radius:6px;'>"
            f"当前阈值: <b>{threshold:.2f}</b> ｜ 高亮残基: <b>{active_count}/{total_count}</b> ｜ 模式: <b>{mode}</b>"
            f"</div>"
        ))

def reload_data(_=None):
    try:
        new_pdb = get_uploaded_text(use_uploaded_pdb) or PDB_TEXT
        new_mmpbsa = get_uploaded_text(use_uploaded_mmpbsa) or MMPBSA_TEXT

        state["pdb_text"] = new_pdb
        state["mmpbsa_text"] = new_mmpbsa
        state["atom_df"] = parse_pdb_atoms(new_pdb)
        state["energy_df"] = parse_mmpbsa_delta_total(new_mmpbsa)
        state["energy_table"] = prepare_energy_table(state["atom_df"], state["energy_df"])

        global PDB_TEMP_PATH
        PDB_TEMP_PATH = make_temp_pdb_file(new_pdb)
        view.clear()
        view.add_component(PDB_TEMP_PATH)
        view.center()

        chains = sorted(state["energy_table"]["chain"].unique().tolist())
        chain_dropdown.options = chains
        chain_dropdown.value = chains[0]
        residue_slider.min = int(state["energy_table"]["resid"].min())
        residue_slider.max = int(state["energy_table"]["resid"].max())
        residue_slider.value = int(state["energy_table"]["resid"].min())
        threshold_slider.max = max(
            0.1,
            float(state["energy_table"]["delta_total"].abs().max())
        )

        refresh_view()
    except Exception as exc:
        with status_output:
            clear_output(wait=True)
            display(HTML(f"<div style='color:#b00020;'><b>数据刷新失败：</b>{exc}</div>"))

reload_button.on_click(reload_data)

for widget_obj in [
    threshold_slider, display_mode, show_backbone, show_sidechain, show_surface,
    chain_dropdown, residue_slider, opacity_slider,
 ]:
    widget_obj.observe(refresh_view, names="value")

## 11. 在 Notebook 中运行完整交互式原型
最后将所有模块整合为一个完整的可交互界面。

> 运行本节后，你将看到：
- 可旋转的蛋白质 3D 结构
- 按 MMPBSA 能量着色的热力显示
- 阈值、模式、透明度与残基选择控件
- 可选的 PDB/MMPBSA 上传入口

In [ ]:
controls_left = widgets.VBox([
    widgets.HTML("<h4 style='margin:0;'>数据输入</h4>"),
    use_uploaded_pdb,
    use_uploaded_mmpbsa,
    reload_button,
    widgets.HTML("<hr style='margin:8px 0;'>"),
    widgets.HTML("<h4 style='margin:0;'>能量与显示</h4>"),
    threshold_slider,
    display_mode,
    opacity_slider,
    widgets.HBox([show_backbone, show_sidechain, show_surface]),
])

controls_right = widgets.VBox([
    widgets.HTML("<h4 style='margin:0;'>残基选择</h4>"),
    chain_dropdown,
    residue_slider,
    widgets.HTML("<hr style='margin:8px 0;'>"),
    widgets.HTML("<h4 style='margin:0;'>状态</h4>"),
    status_output,
    widgets.HTML("<h4 style='margin:8px 0 0 0;'>残基信息</h4>"),
    info_output,
])

dashboard = widgets.HBox([
    widgets.VBox([controls_left, controls_right], layout=widgets.Layout(width="34%")),
    widgets.VBox([view], layout=widgets.Layout(width="66%")),
], layout=widgets.Layout(align_items="flex-start", width="100%"))

reload_data()
display(dashboard)